# RAG over summaries — the collapsed tree joins the index

Node summaries become retrievable exactly like source content: split into paragraphs by
the SAME structure-aware splitter (they're markdown — the grammar applies), stored in the
SAME `paragraphs`/`contents` tables under `source_type='node'`
(`node://hrono/nodes/<id>/<title-slug>@<sig>` — content model, opaque version = the
node's member signature), embedded into the same `(hash, model)` table, indexed by the
same FTS5 — then the FROZEN retrieval recipe runs unchanged over the union.

Hypothesis: architectural questions should now inject summary paragraphs (which know
cross-cutting facts no single chunk states), while detail questions keep hitting source
paragraphs. The old prompts are rerun to check for regressions, plus new architectural
prompts. Node paths carry the node TITLE slug so the α=0.5 path-blend has semantics to
match ("nodes/47/webphone-client-react-frontend", not "nodes/47").

In [ ]:
import json
import re
import sqlite3
import tempfile
from hashlib import sha256
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer

from structuring import split_file_hier, sp_render

APP_DIR = Path.cwd()
DB_PATH = APP_DIR / ".hrono" / "hrono-v2.db"
MODEL_NAME = "ibm-granite/granite-embedding-97m-multilingual-r2"
BATCH = 16

db = sqlite3.connect(DB_PATH)
model = SentenceTransformer(MODEL_NAME)


def slug(title):
    return re.sub(r"[^a-z0-9]+", "-", title.lower()).strip("-")[:60]


# drop-and-rebuild the node-source rows (idempotent re-runs; source rows untouched)
with db:
    db.execute("DELETE FROM paragraphs WHERE source_type = 'node'")

nodes = db.execute("SELECT id, level, sig, summary FROM nodes WHERE summary IS NOT NULL").fetchall()
n_paras = 0
for nid, level, sig, summary in nodes:
    title = summary.lstrip("# ").split("\n")[0]
    path = f"nodes/{nid}/{slug(title)}"
    with tempfile.NamedTemporaryFile(suffix=".md", mode="w", delete=False) as f:
        f.write(summary)
        tmp = Path(f.name)
    _, chunks_ = split_file_hier(tmp)
    raw = tmp.read_bytes()
    tmp.unlink()
    with db:
        for chunk_no, c in enumerate(chunks_):
            for p in c.paragraphs:
                body = raw[p.byte_start:p.byte_end]
                h = sha256(body).hexdigest()
                db.execute("INSERT OR IGNORE INTO contents (hash, body) VALUES (?,?)", (h, body))
                db.execute(
                    "INSERT INTO paragraphs (source_type, instance, path, version, in_chunk,"
                    " byte_start, byte_end, hash, struct_paths, struct_uris)"
                    " VALUES ('node', 'hrono', ?, ?, ?, ?, ?, ?, ?, ?)",
                    (path, sig, chunk_no, p.byte_start, p.byte_end, h,
                     json.dumps(p.struct_paths),
                     " ".join(sp_render(sp) for sp in p.struct_paths)))
                n_paras += 1
print(f"{len(nodes)} summaries -> {n_paras} node paragraphs in the shared tables")

In [ ]:
# embed whatever is new (content-addressed: unchanged summaries cost nothing)
body_of = dict(db.execute(
    "SELECT c.hash, c.body FROM contents c JOIN paragraphs p ON p.hash = c.hash GROUP BY c.hash"))
done = {r[0] for r in db.execute("SELECT hash FROM embeddings WHERE model = ?", (MODEL_NAME,))}
todo = [h for h in body_of if h not in done]
print(f"{len(todo)} new bodies to embed")
for i in range(0, len(todo), BATCH):
    batch = todo[i:i + BATCH]
    vecs = model.encode([body_of[h].decode('utf-8', 'replace') for h in batch],
                        batch_size=BATCH, normalize_embeddings=True, show_progress_bar=False)
    with db:
        db.executemany("INSERT INTO embeddings (hash, model, dims, vector) VALUES (?,?,?,?)",
                       [(h, MODEL_NAME, vecs.shape[1], np.asarray(v, np.float32).tobytes())
                        for h, v in zip(batch, vecs)])

# load the UNION retrieval space: source + node paragraphs, path-blended a=0.5
rows = db.execute("""
    SELECT p.hash, p.path, p.source_type, p.uri, p.struct_uris, e.vector
    FROM paragraphs p JOIN embeddings e ON e.hash = p.hash AND e.model = ?
    GROUP BY p.hash""", (MODEL_NAME,)).fetchall()
p_hashes = [r[0] for r in rows]
p_kind = {r[0]: r[2] for r in rows}
header_by_hash = {r[0]: f"{r[3]} {r[4]}".strip() for r in rows}
P = np.frombuffer(b"".join(r[5] for r in rows), np.float32).reshape(len(rows), -1)

paths = [r[1] for r in rows]
distinct = sorted(set(paths))
pv = dict(zip(distinct, model.encode(distinct, batch_size=BATCH,
                                     normalize_embeddings=True, show_progress_bar=False)))
PATH = np.stack([pv[p] for p in paths])
ALPHA = 0.5
M_BLEND = np.hstack([np.sqrt(1 - ALPHA) * P, np.sqrt(ALPHA) * PATH])

# BM25 over the union
db.executescript("DROP TABLE IF EXISTS para_fts; CREATE VIRTUAL TABLE para_fts USING fts5(text);")
with db:
    db.executemany("INSERT INTO para_fts (rowid, text) VALUES (?,?)",
                   [(pid, f"{path} {body.decode('utf-8', 'replace')}")
                    for pid, path, body in db.execute(
                        "SELECT p.id, p.path, c.body FROM paragraphs p"
                        " JOIN contents c ON c.hash = p.hash")])
hash_by_id = dict(db.execute("SELECT id, hash FROM paragraphs"))
print(f"union space: {len(p_hashes)} paragraphs "
      f"({sum(1 for h in p_hashes if p_kind[h] == 'node')} from summaries)")

## The frozen recipe over the union — old prompts + architectural prompts

Recipe unchanged: Z_VEC 3.5 solo, keyword z ≥ 6 with vector support ≥ 3, weighted RRF
70/30, cap 10, empty → traverse. Hits tagged `[SUMMARY]` / `[source]`.

In [ ]:
Z_VEC, Z_KW, Z_KW_SUPPORT, RRF_K, MAX_K = 3.5, 6.0, 3.0, 60, 10

PROMPTS = [
    # architectural — where summaries should win
    "What is the overall architecture of the webphone application?",
    "What conventions do the webphone React components follow across the codebase?",
    "What are the known cross-cutting hazards and risky patterns in the webphone frontend?",
    "How is logging implemented across the webphone server?",
    "What technologies does the football tournament project use end to end?",
    # detail — source paragraphs should keep winning (regression check)
    "What does ContactService.getByUser return and how is pagination handled?",
    "Which Spring Boot starter dependencies does the football match project use?",
    "How do I run the football match application locally?",
    "How does the webphone client translate its UI into Ukrainian?",
    "Under which license is webphone distributed?",
    "Where is the photo of a contact validated for maximum size?",
    # no-answer controls
    "How is payment processing implemented and which payment provider API is used?",
    "Where is the Kafka topic configuration and how are events consumed?",
]


import pandas as pd
from IPython.display import display, Markdown
pd.set_option("display.max_colwidth", 200)


def short(header):
    return re.sub(r'@[0-9a-f]{40,}', lambda m: m.group(0)[:9], header)


for prompt in PROMPTS:
    q = model.encode([prompt], normalize_embeddings=True)[0]
    qa = np.concatenate([np.sqrt(1 - ALPHA) * q, np.sqrt(ALPHA) * q])
    sims = M_BLEND @ qa
    v_mu, v_sd = float(sims.mean()), float(sims.std())
    order = np.argsort(-sims)
    v_rank = [p_hashes[i] for i in order[:60]]
    sim_by_hash = {p_hashes[i]: float(sims[i]) for i in order[:200]}

    terms = re.findall(r"[A-Za-z0-9_]+", prompt)
    rows = db.execute("SELECT rowid, bm25(para_fts) FROM para_fts WHERE para_fts MATCH ?"
                      " ORDER BY bm25(para_fts) LIMIT 2000", (" OR ".join(terms),)).fetchall()
    best = {}
    for rid, s_ in rows:
        best.setdefault(hash_by_id[rid], -s_)
    ks = np.array(list(best.values())) if best else np.array([0.0])
    k_mu, k_sd = float(ks.mean()), float(ks.std())
    k_rank = list(best)[:60]

    z_v = lambda h: (sim_by_hash[h] - v_mu) / v_sd if h in sim_by_hash else None
    z_k = lambda h: (best[h] - k_mu) / k_sd if h in best and k_sd > 0 else None
    v_qual = {h for h in v_rank[:MAX_K] if z_v(h) is not None and z_v(h) >= Z_VEC}
    k_qual = {h for h in k_rank[:MAX_K]
              if z_k(h) is not None and z_k(h) >= Z_KW
              and z_v(h) is not None and z_v(h) >= Z_KW_SUPPORT}

    def rrf(h):
        s_ = 0.0
        if h in v_rank:
            s_ += 0.7 / (RRF_K + v_rank.index(h) + 1)
        if h in k_rank:
            s_ += 0.3 / (RRF_K + k_rank.index(h) + 1)
        return s_

    final = sorted(v_qual | k_qual, key=rrf, reverse=True)[:MAX_K]
    candidates = sorted(set(v_rank[:MAX_K]) | set(k_rank[:MAX_K]), key=rrf, reverse=True)

    rows_out = []
    for h in candidates:
        sv, zv, sk, zk = sim_by_hash.get(h), z_v(h), best.get(h), z_k(h)
        uri, _, sp = short(header_by_hash[h]).partition(" ")
        rows_out.append({
            "kind": "SUMMARY" if p_kind[h] == "node" else "source",
            "cosine": None if sv is None else round(sv, 3),
            "z_vec": None if zv is None else round(zv, 2),
            "vQ": "Y" if h in v_qual else "",
            "bm25": None if sk is None else round(sk, 1),
            "z_kw": None if zk is None else round(zk, 2),
            "kQ": "Y" if h in k_qual else "",
            "rrf": round(rrf(h), 4),
            "IN": "YES" if h in final else "",
            "file": uri.split("://", 1)[-1].split("@")[0].removeprefix("github.com/branow/").removeprefix("hrono/"),
            "struct": sp,
        })
    df = pd.DataFrame(rows_out)
    n_sum = sum(1 for h in final if p_kind[h] == "node")
    verdict = (f"**INJECT {len(final)}** ({n_sum} from summaries)" if final
               else "**INJECT NOTHING** — consider traversing the tree")
    display(Markdown(
        f"### {prompt}\n"
        f"vector leg: mu={v_mu:.3f} sd={v_sd:.3f} floor={v_mu + Z_VEC * v_sd:.3f} · "
        f"keyword leg: mu={k_mu:.2f} sd={k_sd:.2f} floor={k_mu + Z_KW * k_sd:.2f} · "
        f"Z_VEC={Z_VEC} Z_KW={Z_KW}+support{Z_KW_SUPPORT} → {verdict}"))
    display(df.style.apply(
        lambda r: ["background-color: " + ("#e3f2fd" if r["kind"] == "SUMMARY" else "#e8f5e9")
                   if r["IN"] == "YES" else ""] * len(r), axis=1)
        .format(na_rep="—", precision=3).hide(axis="index"))

## Observations

(To fill after review: do architectural prompts inject summary paragraphs; do detail
prompts regress; do controls still fail safe; does the node-URI scheme hold up.)